In [ ]:
print("=" * 50)
print("ANTES DEL OVERFLOW")
print("=" * 50)

leaf = [10, 20, 30, 40]
print("HOJA:", leaf)

print("\nInsertando 25...")
leaf.append(25)
leaf.sort()

print("OVERFLOW:", leaf)

# Split manual B+ Tree
d = 2
left = leaf[:d]
right = leaf[d:]

separator = right[0]

print("\nDESPUÉS DEL SPLIT")
print("Padre:", [separator])
print("Hoja izquierda:", left)
print("Hoja derecha:", right)
print(f"{separator} se copia al padre")
print(f"Enlace: {left} -> {right}")

ANTES DEL OVERFLOW
HOJA: [10, 20, 30, 40]

Insertando 25...
OVERFLOW: [10, 20, 25, 30, 40]

DESPUÉS DEL SPLIT
Padre: [25]
Hoja izquierda: [10, 20]
Hoja derecha: [25, 30, 40]
25 se copia al padre
Enlace: [10, 20] -> [25, 30, 40]


In [ ]:


class Nodo:
    def __init__(self, es_hoja=False):
        self.claves   = []
        self.hijos    = []      # punteros a hijos (nodos internos) o datos (hojas)
        self.es_hoja  = es_hoja
        self.siguiente = None   # enlace entre hojas

class BPlusTree:
    def __init__(self, d):
        """
        d  = orden del árbol
        Cada nodo puede tener entre d y 2d claves.
        Overflow cuando un nodo llega a 2d+1 claves.
        """
        self.d    = d
        self.raiz = Nodo(es_hoja=True)

    # ----------------------------------------------------------
    # BÚSQUEDA DE LA HOJA correcta para insertar
    # ----------------------------------------------------------
    def _buscar_hoja(self, nodo, clave):
        if nodo.es_hoja:
            return nodo
        for i, c in enumerate(nodo.claves):
            if clave < c:
                return self._buscar_hoja(nodo.hijos[i], clave)
        return self._buscar_hoja(nodo.hijos[-1], clave)

    # ----------------------------------------------------------
    # INSERCIÓN PRINCIPAL
    # ----------------------------------------------------------
    def insertar(self, clave):
        print(f"\n{'='*50}")
        print(f"  Insertando clave: {clave}")
        print(f"{'='*50}")

        resultado = self._insertar_recursivo(self.raiz, clave)

        # Si la raíz se dividió → crear nueva raíz (árbol crece en altura)
        if resultado:
            clave_mediana, nodo_derecho = resultado
            print(f"\n  *** SPLIT DE RAÍZ — nueva raíz con clave {clave_mediana} ***")
            nueva_raiz        = Nodo(es_hoja=False)
            nueva_raiz.claves = [clave_mediana]
            nueva_raiz.hijos  = [self.raiz, nodo_derecho]
            self.raiz         = nueva_raiz

    # ----------------------------------------------------------
    # INSERCIÓN RECURSIVA — retorna (mediana, nodo_der) si hubo split
    # ----------------------------------------------------------
    def _insertar_recursivo(self, nodo, clave):
        if nodo.es_hoja:
            # --- Insertar en hoja en posición ordenada ---
            i = 0
            while i < len(nodo.claves) and clave > nodo.claves[i]:
                i += 1
            nodo.claves.insert(i, clave)

            # ¿Overflow?
            if len(nodo.claves) > 2 * self.d:
                return self._split_hoja(nodo)
            return None

        else:
            # --- Bajar al hijo correcto ---
            i = 0
            while i < len(nodo.claves) and clave >= nodo.claves[i]:
                i += 1
            resultado = self._insertar_recursivo(nodo.hijos[i], clave)

            if resultado:
                clave_mediana, nodo_derecho = resultado
                # Insertar la clave que subió en este nodo interno
                nodo.claves.insert(i, clave_mediana)
                nodo.hijos.insert(i + 1, nodo_derecho)

                # ¿Overflow en nodo interno?
                if len(nodo.claves) > 2 * self.d:
                    return self._split_interno(nodo)
            return None

    # ----------------------------------------------------------
    # SPLIT DE HOJA — la clave del medio se COPIA al padre
    # ----------------------------------------------------------
    def _split_hoja(self, hoja):
        mid          = len(hoja.claves) // 2
        clave_subida = hoja.claves[mid]   # se COPIA (queda en la hoja también)

        nueva_hoja        = Nodo(es_hoja=True)
        nueva_hoja.claves = hoja.claves[mid:]
        hoja.claves       = hoja.claves[:mid]

        # Mantener lista enlazada de hojas
        nueva_hoja.siguiente = hoja.siguiente
        hoja.siguiente       = nueva_hoja

        print(f"  → SPLIT DE HOJA")
        print(f"     Hoja izquierda : {hoja.claves}")
        print(f"     Hoja derecha   : {nueva_hoja.claves}")
        print(f"     Clave que SUBE (copia): {clave_subida}")

        return clave_subida, nueva_hoja

    # ----------------------------------------------------------
    # SPLIT DE NODO INTERNO — la clave del medio SUBE (no se copia)
    # ----------------------------------------------------------
    def _split_interno(self, nodo):
        mid          = len(nodo.claves) // 2
        clave_subida = nodo.claves[mid]   # SUBE y desaparece del nivel actual

        nuevo_nodo        = Nodo(es_hoja=False)
        nuevo_nodo.claves = nodo.claves[mid + 1:]
        nuevo_nodo.hijos  = nodo.hijos[mid + 1:]

        nodo.claves = nodo.claves[:mid]
        nodo.hijos  = nodo.hijos[:mid + 1]

        print(f"  → SPLIT DE NODO INTERNO")
        print(f"     Nodo izquierdo : {nodo.claves}")
        print(f"     Nodo derecho   : {nuevo_nodo.claves}")
        print(f"     Clave que SUBE (desaparece del nivel): {clave_subida}")

        return clave_subida, nuevo_nodo

    # ----------------------------------------------------------
    # IMPRIMIR el árbol nivel por nivel
    # ----------------------------------------------------------
    def imprimir(self):
        print("\n  Estructura del árbol:")
        niveles = [[(self.raiz, "raíz")]]
        while True:
            siguiente_nivel = []
            for nodo, etiqueta in niveles[-1]:
                if not nodo.es_hoja:
                    for hijo in nodo.hijos:
                        tipo = "hoja" if hijo.es_hoja else "interno"
                        siguiente_nivel.append((hijo, tipo))
            if not siguiente_nivel:
                break
            niveles.append(siguiente_nivel)

        for i, nivel in enumerate(niveles):
            linea = f"  Nivel {i}: "
            for nodo, etiqueta in nivel:
                linea += f"[{' | '.join(map(str, nodo.claves))}]({etiqueta})  "
            print(linea)

        # Mostrar lista enlazada de hojas
        hojas = []
        nodo  = self.raiz
        while not nodo.es_hoja:
            nodo = nodo.hijos[0]
        while nodo:
            hojas.append(str(nodo.claves))
            nodo = nodo.siguiente
        print(f"  Hojas enlazadas: {' → '.join(hojas)}")


# ============================================================
#  DEMOSTRACIÓN
#  Secuencia diseñada para provocar los 3 tipos de split:
#    split de hoja → split de nodo interno → split de raíz
# ============================================================

arbol = BPlusTree(d=2)   # máximo 4 claves por nodo, overflow con 5

secuencia = [10, 20, 5, 30, 40, 3, 25, 15, 35, 50]

print("B+ Tree — Inserción y Split")
print(f"Orden d=2 | Máx claves por nodo: {2*arbol.d} | Overflow con: {2*arbol.d+1}")
print(f"Secuencia a insertar: {secuencia}")

for clave in secuencia:
    arbol.insertar(clave)
    arbol.imprimir()

B+ Tree — Inserción y Split
Orden d=2 | Máx claves por nodo: 4 | Overflow con: 5
Secuencia a insertar: [10, 20, 5, 30, 40, 3, 25, 15, 35, 50]

  Insertando clave: 10

  Estructura del árbol:
  Nivel 0: [10](raíz)  
  Hojas enlazadas: [10]

  Insertando clave: 20

  Estructura del árbol:
  Nivel 0: [10 | 20](raíz)  
  Hojas enlazadas: [10, 20]

  Insertando clave: 5

  Estructura del árbol:
  Nivel 0: [5 | 10 | 20](raíz)  
  Hojas enlazadas: [5, 10, 20]

  Insertando clave: 30

  Estructura del árbol:
  Nivel 0: [5 | 10 | 20 | 30](raíz)  
  Hojas enlazadas: [5, 10, 20, 30]

  Insertando clave: 40
  → SPLIT DE HOJA
     Hoja izquierda : [5, 10]
     Hoja derecha   : [20, 30, 40]
     Clave que SUBE (copia): 20

  *** SPLIT DE RAÍZ — nueva raíz con clave 20 ***

  Estructura del árbol:
  Nivel 0: [20](raíz)  
  Nivel 1: [5 | 10](hoja)  [20 | 30 | 40](hoja)  
  Hojas enlazadas: [5, 10] → [20, 30, 40]

  Insertando clave: 3

  Estructura del árbol:
  Nivel 0: [20](raíz)  
  Nivel 1: [3 | 5 